# Setup

In [ ]:
library(miloR)
library(SingleCellExperiment)
library(zellkonverter)
library(dplyr)
library(patchwork)
library(ggplot2)
library(scater)
library(ggtext)
library(readr)
library(tibble)
library(forcats)
library(Seurat)
library(stringr)

setZellkonverterVerbose(TRUE)

colCTRL <- "#1E88E5"
colFGID <- "#1E88E5"
coliCD <- "#55D276"
colcCD <- "#FFC107"
colUC <- "#D81B60"
colIBD <- "#EB6D33"

`%notin%` <- Negate(`%in%`)

alphaVal <- 0.10

thisseed <- 231109

# ColEpi

## Pre-Processing

In [ ]:
thiscelltype <- "colepi"

In [ ]:
sce.colepi <- readH5AD(
    sprintf('../../predict/data/objects/annotated/predictCOL__CB_%s_annotated_231226.h5ad', thiscelltype)
)
sce.colepi

In [ ]:
sce.colepi <- subset(sce.colepi, , timepoint == "T0" & diagSub != "iCD")
sce.colepi

In [ ]:
sce.colepi$diagSub <- droplevels(sce.colepi$diagSub)
sce.colepi$timepoint <- droplevels(sce.colepi$timepoint)
sce.colepi$group <- droplevels(sce.colepi$group)

In [ ]:
alldf.colepi <- data.frame(barcode = colnames(sce.colepi), diagSub = sce.colepi$diagSub)
alldf.colepi

In [ ]:
groupCount.colepi <- alldf.colepi %>% pull(diagSub) %>% table
groupCount.colepi

In [ ]:
dsVal.colepi <- groupCount.colepi %>% min
dsVal.colepi

In [ ]:
set.seed(thisseed)
dsdf.colepi <- alldf.colepi %>%
    group_by(diagSub) %>% 
    sample_n(dsVal.colepi, )

dsdf.colepi

In [ ]:
dsdf.colepi %>% pull(diagSub) %>% table

In [ ]:
# perform downsample
sce.colepi <- subset(sce.colepi, , colnames(sce.colepi) %in% dsdf.colepi$barcode)
sce.colepi

In [ ]:
sce.colepi$biopsy <- droplevels(sce.colepi$biopsy)

In [ ]:
num_cells <- sort(table(sce.colepi$biopsy))
num_cells

In [ ]:
filter_out <- num_cells[num_cells < 5] %>% names
ifelse(identical(filter_out, character(0)), 
       print("No Biopsies Removed"), 
       filter_out)

In [ ]:
# remove biopsies w/ fewer than 5 cells
sce.colepi <- subset(sce.colepi, , biopsy %notin% filter_out)
sce.colepi

In [ ]:
sce.colepi$biopsy <- droplevels(sce.colepi$biopsy)

In [ ]:
sort(table(sce.colepi$biopsy))

In [ ]:
n_distinct(sce.colepi$biopsy)

In [ ]:
milo.colepi <- Milo(sce.colepi)
milo.colepi

In [ ]:
k.colepi <- 200
d.colepi <- 30
prop.colepi <- 0.025

In [ ]:
milo.colepi <- buildGraph(milo.colepi, k = k.colepi, d = d.colepi, reduced.dim = "X_scVI")
milo.colepi

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=120)
milo.colepi <- makeNhoods(milo.colepi, prop = prop.colepi, k = k.colepi, d=d.colepi, refined = TRUE, reduced_dims = "X_scVI")
plotNhoodSizeHist(milo.colepi)

In [ ]:
milo.colepi <- countCells(milo.colepi, meta.data = data.frame(colData(milo.colepi)), samples="biopsy")
head(nhoodCounts(milo.colepi))

In [ ]:
dim(nhoodCounts(milo.colepi))

In [ ]:
nhoodCounts(milo.colepi) %>% rowSums %>% range

In [ ]:
nhoodCounts(milo.colepi) %>% rowSums %>% range %>% .[1]/dim(nhoodCounts(milo.colepi))[2]

In [ ]:
nhoodCounts(milo.colepi) %>% rowSums %>% mean

In [ ]:
nhoodCounts(milo.colepi) %>% rowSums %>% median

In [ ]:
dim(nhoodCounts(milo.colepi))[2]*5

In [ ]:
nhoodCounts(milo.colepi) %>% rowSums %>% mean > dim(nhoodCounts(milo.colepi))[2]*5

In [ ]:
nhoodCounts(milo.colepi) %>% colSums %>% sort

In [ ]:
milo.colepi <- calcNhoodDistance(milo.colepi, d=d.colepi, reduced.dim = "X_scVI")

In [ ]:
milo.colepi <- buildNhoodGraph(milo.colepi)

## Design

In [ ]:
diagSub.vector <- colData(milo.colepi) %>%
    as.tibble %>%
    mutate(diagSub = fct_relevel(diagSub, c("FGID", "iCD", "cCD", "UC"))) %>%
    pull(diagSub)

In [ ]:
group.vector <- colData(milo.colepi) %>%
    as.tibble %>%
    mutate(group = fct_relevel(group, c("CTRL", "ILEAL", "IBD"))) %>%
    pull(group)

In [ ]:
biopsy.vector <- colData(milo.colepi) %>%
    as.tibble %>%
    mutate(biopsy = as.factor(biopsy)) %>%
    pull(biopsy)

In [ ]:
version_10x.vector <- colData(milo.colepi) %>%
    as.tibble %>%
    mutate(version_10x = as.factor(version_10x)) %>%
    pull(version_10x)

In [ ]:
milo.colepi$diagSub <- diagSub.vector
milo.colepi$group <- group.vector
milo.colepi$biopsy <- biopsy.vector
milo.colepi$version_10x <- version_10x.vector

In [ ]:
design.colepi <- data.frame(colData(milo.colepi))[,c("biopsy", "diagSub", "group", "version_10x")]
design.colepi <- distinct(design.colepi)

rownames(design.colepi) <- design.colepi$biopsy

## Reorder rownames to match columns of nhoodCounts(milo)
design.colepi <- design.colepi[colnames(nhoodCounts(milo.colepi)), , drop=FALSE]

head(design.colepi)
levels(design.colepi$diagSub)
levels(design.colepi$group)
levels(design.colepi$version_10x)

## Prepare Seurat Objects

This is for plotting purposes

In [ ]:
seuobj.colepi <- as.Seurat(milo.colepi, counts = "counts", data = NULL)
seuobj.colepi

In [ ]:
seuobj.colepi <- AddMetaData(seuobj.colepi, seuobj.colepi@reductions$X_umap@cell.embeddings %>% as.data.frame)
seuobj.colepi@meta.data

## Set Plot Limits

In [ ]:
topX <- 23.5
topY <- 12.5

bottomX <- 23.5
bottomY <- -9.75

midX <- 26.75
midY <- 1
midLabel <- "**LogFC**"

limsX <- c(-3.5,20.5)
limsY <- c(-9.5, 13)

## IBD vs. CTRL

In [ ]:
design.colepi.ctrlibd <- design.colepi %>%
    filter(group %in% c("CTRL", "IBD")) %>%
    droplevels

design.colepi.ctrlibd

In [ ]:
da_results.colepi.ctrlibd <- testNhoods(milo.colepi, design = ~version_10x + group, design.df = design.colepi.ctrlibd, 
#                                  model.contrasts=c("groupIBD - groupCTRL"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.colepi.ctrlibd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.colepi.ctrlibd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.colepi.ctrlibd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.colepi.ctrlibd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.colepi, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.colepi, da_results.colepi.ctrlibd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.colepi.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.colepi, da_results.colepi.ctrlibd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "ColEpi Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
thiscelltype

In [ ]:
da_results.colepi.ctrlibd.indexed <- da_results.colepi.ctrlibd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.colepi))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.colepi) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.colepi@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.colepi.ctrlibd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.colepi.ctrlibd <- annotateNhoods(milo.colepi, da_results.colepi.ctrlibd, coldata_col = "cellstate")
head(da_results.colepi.ctrlibd)

In [ ]:
celltypeorder.ccduc <- da_results.colepi.ctrlibd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.colepi.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colCTRL
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.colepi.ctrlibd$cellstate)+0.5)

plotDAbeeswarm(da_results.colepi.ctrlibd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. cCD

In [ ]:
design.colepi.ccduc <- design.colepi %>%
    filter(diagSub %in% c("UC", "cCD")) %>%
    droplevels

design.colepi.ccduc

In [ ]:
da_results.colepi.ccduc <- testNhoods(milo.colepi, design = ~version_10x + diagSub, design.df = design.colepi.ccduc, 
#                                  model.contrasts=c("diagSubUC - diagSubcCD"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.colepi.ccduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.colepi.ccduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.colepi.ccduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.colepi.ccduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.colepi, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.colepi, da_results.colepi.ccduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.colepi.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD

plotNhoodGraphDA(milo.colepi, da_results.colepi.ccduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "ColEpi Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.colepi.ccduc.indexed <- da_results.colepi.ccduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.colepi))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.colepi) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.colepi@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.colepi.ccduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.colepi.ccduc <- annotateNhoods(milo.colepi, da_results.colepi.ccduc, coldata_col = "cellstate")
head(da_results.colepi.ccduc)

In [ ]:
celltypeorder.ccduc <- da_results.colepi.ccduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.colepi.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.colepi.ccduc$cellstate)+0.5)

plotDAbeeswarm(da_results.colepi.ccduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. FGID

In [ ]:
design.colepi.fgiduc <- design.colepi %>%
    filter(diagSub %in% c("FGID", "UC")) %>%
    droplevels

design.colepi.fgiduc

In [ ]:
da_results.colepi.fgiduc <- testNhoods(milo.colepi, design = ~version_10x + diagSub, design.df = design.colepi.fgiduc, 
#                                  model.contrasts=c("diagSubUC - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.colepi.fgiduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.colepi.fgiduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.colepi.fgiduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.colepi.fgiduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.colepi, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.colepi, da_results.colepi.fgiduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.colepi.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.colepi, da_results.colepi.fgiduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "ColEpi Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.colepi.fgiduc.indexed <- da_results.colepi.fgiduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.colepi))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.colepi) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.colepi@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.colepi.fgiduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.colepi.fgiduc <- annotateNhoods(milo.colepi, da_results.colepi.fgiduc, coldata_col = "cellstate")
head(da_results.colepi.fgiduc)

In [ ]:
celltypeorder.fgiduc <- da_results.colepi.fgiduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.colepi.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.colepi.fgiduc$cellstate)+0.5)

plotDAbeeswarm(da_results.colepi.fgiduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgiduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## cCD vs. FGID

In [ ]:
design.colepi.fgidccd <- design.colepi %>%
    filter(diagSub %in% c("FGID", "cCD")) %>%
    droplevels

design.colepi.fgidccd

In [ ]:
da_results.colepi.fgidccd <- testNhoods(milo.colepi, design = ~version_10x + diagSub, design.df = design.colepi.fgidccd, 
#                                  model.contrasts=c("diagSubcCD - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.colepi.fgidccd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.colepi.fgidccd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.colepi.fgidccd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.colepi.fgidccd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.colepi, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.colepi, da_results.colepi.fgidccd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.colepi.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.colepi, da_results.colepi.fgidccd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "ColEpi Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.colepi.fgidccd.indexed <- da_results.colepi.fgidccd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.colepi))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.colepi) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.colepi@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.colepi.fgidccd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (cCD vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.colepi.fgidccd <- annotateNhoods(milo.colepi, da_results.colepi.fgidccd, coldata_col = "cellstate")
head(da_results.colepi.fgidccd)

In [ ]:
celltypeorder.fgidccd <- da_results.colepi.fgidccd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.colepi.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.colepi.fgidccd$cellstate)+0.5)

plotDAbeeswarm(da_results.colepi.fgidccd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgidccd))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## Saves

### DA Results

In [ ]:
write_csv(da_results.colepi.ctrlibd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ctrlibd.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.colepi.ccduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ccduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.colepi.fgiduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgiduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.colepi.fgidccd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgidccd.csv", 
                 thiscelltype, thiscelltype)
         )

### Neighbourhoods

In [ ]:
Matrix::writeMM(nhoods(milo.colepi), 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_sparse.txt", 
                thiscelltype, thiscelltype)
       )

In [ ]:
write_tsv(as.data.frame(as.matrix(nhoodCounts(milo.colepi))) %>% rownames_to_column("nhood"),
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_dense.txt", 
                thiscelltype, thiscelltype)
       )

### Milo Object

In [ ]:
milo.colepi

In [ ]:
saveRDS(milo.colepi, 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x.rds", thiscelltype, thiscelltype)
       )

# Stroma

## Pre-Processing

In [ ]:
thiscelltype <- "stroma"

In [ ]:
sce.stroma <- readH5AD(
    sprintf('../../predict/data/objects/annotated/predictCOL__CB_%s_annotated_231226.h5ad', thiscelltype)
)
sce.stroma

In [ ]:
sce.stroma <- subset(sce.stroma, , timepoint == "T0" & diagSub != "iCD")
sce.stroma

In [ ]:
sce.stroma$diagSub <- droplevels(sce.stroma$diagSub)
sce.stroma$timepoint <- droplevels(sce.stroma$timepoint)
sce.stroma$group <- droplevels(sce.stroma$group)

In [ ]:
alldf.stroma <- data.frame(barcode = colnames(sce.stroma), diagSub = sce.stroma$diagSub)
alldf.stroma

In [ ]:
groupCount.stroma <- alldf.stroma %>% pull(diagSub) %>% table
groupCount.stroma

In [ ]:
dsVal.stroma <- groupCount.stroma %>% min
dsVal.stroma

In [ ]:
set.seed(thisseed)
dsdf.stroma <- alldf.stroma %>%
    group_by(diagSub) %>% 
    sample_n(dsVal.stroma, )

dsdf.stroma

In [ ]:
dsdf.stroma %>% pull(diagSub) %>% table

In [ ]:
# perform downsample
sce.stroma <- subset(sce.stroma, , colnames(sce.stroma) %in% dsdf.stroma$barcode)
sce.stroma

In [ ]:
sce.stroma$biopsy <- droplevels(sce.stroma$biopsy)

In [ ]:
num_cells <- sort(table(sce.stroma$biopsy))
num_cells

In [ ]:
filter_out <- num_cells[num_cells < 5] %>% names
ifelse(identical(filter_out, character(0)), 
       print("No Biopsies Removed"), 
       filter_out)

In [ ]:
# remove biopsies w/ fewer than 5 cells
sce.stroma <- subset(sce.stroma, , biopsy %notin% filter_out)
sce.stroma

In [ ]:
sce.stroma$biopsy <- droplevels(sce.stroma$biopsy)

In [ ]:
sort(table(sce.stroma$biopsy))

In [ ]:
n_distinct(sce.stroma$biopsy)

In [ ]:
milo.stroma <- Milo(sce.stroma)
milo.stroma

In [ ]:
k.stroma <- 200
d.stroma <- 30
prop.stroma <- 0.05

In [ ]:
milo.stroma <- buildGraph(milo.stroma, k = k.stroma, d = d.stroma, reduced.dim = "X_scVI")
milo.stroma

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=120)
milo.stroma <- makeNhoods(milo.stroma, prop = prop.stroma, k = k.stroma, d=d.stroma, refined = TRUE, reduced_dims = "X_scVI")
plotNhoodSizeHist(milo.stroma)

In [ ]:
milo.stroma <- countCells(milo.stroma, meta.data = data.frame(colData(milo.stroma)), samples="biopsy")
head(nhoodCounts(milo.stroma))

In [ ]:
dim(nhoodCounts(milo.stroma))

In [ ]:
nhoodCounts(milo.stroma) %>% rowSums %>% range

In [ ]:
nhoodCounts(milo.stroma) %>% rowSums %>% range %>% .[1]/dim(nhoodCounts(milo.stroma))[2]

In [ ]:
nhoodCounts(milo.stroma) %>% rowSums %>% mean

In [ ]:
nhoodCounts(milo.stroma) %>% rowSums %>% median

In [ ]:
dim(nhoodCounts(milo.stroma))[2]*5

In [ ]:
nhoodCounts(milo.stroma) %>% rowSums %>% mean > dim(nhoodCounts(milo.stroma))[2]*5

In [ ]:
nhoodCounts(milo.stroma) %>% colSums %>% sort

In [ ]:
milo.stroma <- calcNhoodDistance(milo.stroma, d=d.stroma, reduced.dim = "X_scVI")

In [ ]:
milo.stroma <- buildNhoodGraph(milo.stroma)

## Design

In [ ]:
diagSub.vector <- colData(milo.stroma) %>%
    as.tibble %>%
    mutate(diagSub = fct_relevel(diagSub, c("FGID", "iCD", "cCD", "UC"))) %>%
    pull(diagSub)

In [ ]:
group.vector <- colData(milo.stroma) %>%
    as.tibble %>%
    mutate(group = fct_relevel(group, c("CTRL", "ILEAL", "IBD"))) %>%
    pull(group)

In [ ]:
biopsy.vector <- colData(milo.stroma) %>%
    as.tibble %>%
    mutate(biopsy = as.factor(biopsy)) %>%
    pull(biopsy)

In [ ]:
version_10x.vector <- colData(milo.stroma) %>%
    as.tibble %>%
    mutate(version_10x = as.factor(version_10x)) %>%
    pull(version_10x)

In [ ]:
milo.stroma$diagSub <- diagSub.vector
milo.stroma$group <- group.vector
milo.stroma$biopsy <- biopsy.vector
milo.stroma$version_10x <- version_10x.vector

In [ ]:
design.stroma <- data.frame(colData(milo.stroma))[,c("biopsy", "diagSub", "group", "version_10x")]
design.stroma <- distinct(design.stroma)

rownames(design.stroma) <- design.stroma$biopsy

## Reorder rownames to match columns of nhoodCounts(milo)
design.stroma <- design.stroma[colnames(nhoodCounts(milo.stroma)), , drop=FALSE]

head(design.stroma)
levels(design.stroma$diagSub)
levels(design.stroma$group)
levels(design.stroma$version_10x)

## Prepare Seurat Objects

This is for plotting purposes

In [ ]:
seuobj.stroma <- as.Seurat(milo.stroma, counts = "counts", data = NULL)
seuobj.stroma

In [ ]:
seuobj.stroma <- AddMetaData(seuobj.stroma, seuobj.stroma@reductions$X_umap@cell.embeddings %>% as.data.frame)
seuobj.stroma@meta.data

## Set Plot Limits

In [ ]:
topX <- 15.5
topY <- 15.5

bottomX <- 15.5
bottomY <- -1

midX <- 18
midY <- 7.5
midLabel <- "**LogFC**"

limsX <- c(-7,13)
limsY <- c(-1, 16)

## IBD vs. CTRL

In [ ]:
design.stroma.ctrlibd <- design.stroma %>%
    filter(group %in% c("CTRL", "IBD")) %>%
    droplevels

design.stroma.ctrlibd

In [ ]:
da_results.stroma.ctrlibd <- testNhoods(milo.stroma, design = ~version_10x + group, design.df = design.stroma.ctrlibd, 
#                                  model.contrasts=c("groupIBD - groupCTRL"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.stroma.ctrlibd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.stroma.ctrlibd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.stroma.ctrlibd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.stroma.ctrlibd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.stroma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.stroma, da_results.stroma.ctrlibd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.stroma.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.stroma, da_results.stroma.ctrlibd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Stroma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.ctrlibd.indexed <- da_results.stroma.ctrlibd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.stroma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.stroma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.stroma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.stroma.ctrlibd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.ctrlibd <- annotateNhoods(milo.stroma, da_results.stroma.ctrlibd, coldata_col = "cellstate")
head(da_results.stroma.ctrlibd)

In [ ]:
celltypeorder.ccduc <- da_results.stroma.ctrlibd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.stroma.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colCTRL
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.stroma.ctrlibd$cellstate)+0.5)

plotDAbeeswarm(da_results.stroma.ctrlibd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. cCD

In [ ]:
design.stroma.ccduc <- design.stroma %>%
    filter(diagSub %in% c("UC", "cCD")) %>%
    droplevels

design.stroma.ccduc

In [ ]:
da_results.stroma.ccduc <- testNhoods(milo.stroma, design = ~version_10x + diagSub, design.df = design.stroma.ccduc, 
#                                  model.contrasts=c("diagSubUC - diagSubcCD"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.stroma.ccduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.stroma.ccduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.stroma.ccduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.stroma.ccduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.stroma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.stroma, da_results.stroma.ccduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.stroma.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD

plotNhoodGraphDA(milo.stroma, da_results.stroma.ccduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Stroma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.ccduc.indexed <- da_results.stroma.ccduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.stroma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.stroma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.stroma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.stroma.ccduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.ccduc <- annotateNhoods(milo.stroma, da_results.stroma.ccduc, coldata_col = "cellstate")
head(da_results.stroma.ccduc)

In [ ]:
celltypeorder.ccduc <- da_results.stroma.ccduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.stroma.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.stroma.ccduc$cellstate)+0.5)

plotDAbeeswarm(da_results.stroma.ccduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. FGID

In [ ]:
design.stroma.fgiduc <- design.stroma %>%
    filter(diagSub %in% c("FGID", "UC")) %>%
    droplevels

design.stroma.fgiduc

In [ ]:
da_results.stroma.fgiduc <- testNhoods(milo.stroma, design = ~version_10x + diagSub, design.df = design.stroma.fgiduc, 
#                                  model.contrasts=c("diagSubUC - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.stroma.fgiduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.stroma.fgiduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.stroma.fgiduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.stroma.fgiduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.stroma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.stroma, da_results.stroma.fgiduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.stroma.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.stroma, da_results.stroma.fgiduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Stroma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.fgiduc.indexed <- da_results.stroma.fgiduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.stroma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.stroma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.stroma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.stroma.fgiduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.fgiduc <- annotateNhoods(milo.stroma, da_results.stroma.fgiduc, coldata_col = "cellstate")
head(da_results.stroma.fgiduc)

In [ ]:
celltypeorder.fgiduc <- da_results.stroma.fgiduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.stroma.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.stroma.fgiduc$cellstate)+0.5)

plotDAbeeswarm(da_results.stroma.fgiduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgiduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## cCD vs. FGID

In [ ]:
design.stroma.fgidccd <- design.stroma %>%
    filter(diagSub %in% c("FGID", "cCD")) %>%
    droplevels

design.stroma.fgidccd

In [ ]:
da_results.stroma.fgidccd <- testNhoods(milo.stroma, design = ~version_10x + diagSub, design.df = design.stroma.fgidccd, 
#                                  model.contrasts=c("diagSubcCD - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.stroma.fgidccd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.stroma.fgidccd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.stroma.fgidccd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.stroma.fgidccd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.stroma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.stroma, da_results.stroma.fgidccd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.stroma.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.stroma, da_results.stroma.fgidccd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Stroma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.fgidccd.indexed <- da_results.stroma.fgidccd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.stroma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.stroma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.stroma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.stroma.fgidccd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (cCD vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.stroma.fgidccd <- annotateNhoods(milo.stroma, da_results.stroma.fgidccd, coldata_col = "cellstate")
head(da_results.stroma.fgidccd)

In [ ]:
celltypeorder.fgidccd <- da_results.stroma.fgidccd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.stroma.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.stroma.fgidccd$cellstate)+0.5)

plotDAbeeswarm(da_results.stroma.fgidccd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgidccd))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## Saves

### DA Results

In [ ]:
write_csv(da_results.stroma.ctrlibd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ctrlibd.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.stroma.ccduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ccduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.stroma.fgiduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgiduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.stroma.fgidccd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgidccd.csv", 
                 thiscelltype, thiscelltype)
         )

### Neighbourhoods

In [ ]:
Matrix::writeMM(nhoods(milo.stroma), 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_sparse.txt", 
                thiscelltype, thiscelltype)
       )

In [ ]:
write_tsv(as.data.frame(as.matrix(nhoodCounts(milo.stroma))) %>% rownames_to_column("nhood"),
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_dense.txt", 
                thiscelltype, thiscelltype)
       )

### Milo Object

In [ ]:
milo.stroma

In [ ]:
saveRDS(milo.stroma, 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x.rds", thiscelltype, thiscelltype)
       )

# Myeloid

## Pre-Processing

In [ ]:
thiscelltype <- "myeloid"

In [ ]:
sce.myeloid <- readH5AD(
    sprintf('../../predict/data/objects/annotated/predictCOL__CB_%s_annotated_231226.h5ad', thiscelltype)
)
sce.myeloid

In [ ]:
sce.myeloid <- subset(sce.myeloid, , timepoint == "T0" & diagSub != "iCD")
sce.myeloid

In [ ]:
sce.myeloid$diagSub <- droplevels(sce.myeloid$diagSub)
sce.myeloid$timepoint <- droplevels(sce.myeloid$timepoint)
sce.myeloid$group <- droplevels(sce.myeloid$group)

In [ ]:
alldf.myeloid <- data.frame(barcode = colnames(sce.myeloid), diagSub = sce.myeloid$diagSub)
alldf.myeloid

In [ ]:
groupCount.myeloid <- alldf.myeloid %>% pull(diagSub) %>% table
groupCount.myeloid

In [ ]:
dsVal.myeloid <- groupCount.myeloid %>% min
dsVal.myeloid

In [ ]:
set.seed(thisseed)
dsdf.myeloid <- alldf.myeloid %>%
    group_by(diagSub) %>% 
    sample_n(dsVal.myeloid, )

dsdf.myeloid

In [ ]:
dsdf.myeloid %>% pull(diagSub) %>% table

In [ ]:
# perform downsample
sce.myeloid <- subset(sce.myeloid, , colnames(sce.myeloid) %in% dsdf.myeloid$barcode)
sce.myeloid

In [ ]:
sce.myeloid$biopsy <- droplevels(sce.myeloid$biopsy)

In [ ]:
num_cells <- sort(table(sce.myeloid$biopsy))
num_cells

In [ ]:
filter_out <- num_cells[num_cells < 5] %>% names
ifelse(identical(filter_out, character(0)), 
       print("No Biopsies Removed"), 
       filter_out)

In [ ]:
# remove biopsies w/ fewer than 5 cells
sce.myeloid <- subset(sce.myeloid, , biopsy %notin% filter_out)
sce.myeloid

In [ ]:
sce.myeloid$biopsy <- droplevels(sce.myeloid$biopsy)

In [ ]:
sort(table(sce.myeloid$biopsy))

In [ ]:
n_distinct(sce.myeloid$biopsy)

In [ ]:
milo.myeloid <- Milo(sce.myeloid)
milo.myeloid

In [ ]:
k.myeloid <- 200
d.myeloid <- 30
prop.myeloid <- 0.05

In [ ]:
milo.myeloid <- buildGraph(milo.myeloid, k = k.myeloid, d = d.myeloid, reduced.dim = "X_scVI")
milo.myeloid

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=120)
milo.myeloid <- makeNhoods(milo.myeloid, prop = prop.myeloid, k = k.myeloid, d=d.myeloid, refined = TRUE, reduced_dims = "X_scVI")
plotNhoodSizeHist(milo.myeloid)

In [ ]:
milo.myeloid <- countCells(milo.myeloid, meta.data = data.frame(colData(milo.myeloid)), samples="biopsy")
head(nhoodCounts(milo.myeloid))

Confirm choice of k allows for nhoods > 5 x # of samples 

In [ ]:
dim(nhoodCounts(milo.myeloid))

In [ ]:
nhoodCounts(milo.myeloid) %>% rowSums %>% range

In [ ]:
nhoodCounts(milo.myeloid) %>% rowSums %>% range %>% .[1]/dim(nhoodCounts(milo.myeloid))[2]

In [ ]:
nhoodCounts(milo.myeloid) %>% rowSums %>% mean

In [ ]:
nhoodCounts(milo.myeloid) %>% rowSums %>% median

In [ ]:
dim(nhoodCounts(milo.myeloid))[2]*5

In [ ]:
nhoodCounts(milo.myeloid) %>% rowSums %>% mean > dim(nhoodCounts(milo.myeloid))[2]*5

In [ ]:
nhoodCounts(milo.myeloid) %>% colSums %>% sort

In [ ]:
milo.myeloid <- calcNhoodDistance(milo.myeloid, d=d.myeloid, reduced.dim = "X_scVI")

In [ ]:
milo.myeloid <- buildNhoodGraph(milo.myeloid)

## Design

In [ ]:
diagSub.vector <- colData(milo.myeloid) %>%
    as.tibble %>%
    mutate(diagSub = fct_relevel(diagSub, c("FGID", "iCD", "cCD", "UC"))) %>%
    pull(diagSub)

In [ ]:
group.vector <- colData(milo.myeloid) %>%
    as.tibble %>%
    mutate(group = fct_relevel(group, c("CTRL", "ILEAL", "IBD"))) %>%
    pull(group)

In [ ]:
biopsy.vector <- colData(milo.myeloid) %>%
    as.tibble %>%
    mutate(biopsy = as.factor(biopsy)) %>%
    pull(biopsy)

In [ ]:
version_10x.vector <- colData(milo.myeloid) %>%
    as.tibble %>%
    mutate(version_10x = as.factor(version_10x)) %>%
    pull(version_10x)

In [ ]:
milo.myeloid$diagSub <- diagSub.vector
milo.myeloid$group <- group.vector
milo.myeloid$biopsy <- biopsy.vector
milo.myeloid$version_10x <- version_10x.vector

In [ ]:
design.myeloid <- data.frame(colData(milo.myeloid))[,c("biopsy", "diagSub", "group", "version_10x")]
design.myeloid <- distinct(design.myeloid)

rownames(design.myeloid) <- design.myeloid$biopsy

## Reorder rownames to match columns of nhoodCounts(milo)
design.myeloid <- design.myeloid[colnames(nhoodCounts(milo.myeloid)), , drop=FALSE]

head(design.myeloid)
levels(design.myeloid$diagSub)
levels(design.myeloid$group)
levels(design.myeloid$version_10x)

## Prepare Seurat Objects

This is for plotting purposes

In [ ]:
seuobj.myeloid <- as.Seurat(milo.myeloid, counts = "counts", data = NULL)
seuobj.myeloid

In [ ]:
seuobj.myeloid <- AddMetaData(seuobj.myeloid, seuobj.myeloid@reductions$X_umap@cell.embeddings %>% as.data.frame)
seuobj.myeloid@meta.data

## Set Plot Limits

In [ ]:
topX <- 18
topY <- 12.5

bottomX <- 18
bottomY <- -8.5

midX <- 20.5
midY <- 2
midLabel <- "**LogFC**"

limsX <- c(-5.5,15.5)
limsY <- c(-8.5, 13)

## IBD vs. CTRL

In [ ]:
design.myeloid.ctrlibd <- design.myeloid %>%
    filter(group %in% c("CTRL", "IBD")) %>%
    droplevels

design.myeloid.ctrlibd

In [ ]:
da_results.myeloid.ctrlibd <- testNhoods(milo.myeloid, design = ~version_10x + group, design.df = design.myeloid.ctrlibd, 
#                                  model.contrasts=c("groupIBD - groupCTRL"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.myeloid.ctrlibd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.myeloid.ctrlibd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.myeloid.ctrlibd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.myeloid.ctrlibd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.myeloid, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.ctrlibd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.myeloid.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.ctrlibd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Myeloid Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
thiscelltype

In [ ]:
da_results.myeloid.ctrlibd.indexed <- da_results.myeloid.ctrlibd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.myeloid))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.myeloid) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.myeloid@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.myeloid.ctrlibd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.myeloid.ctrlibd <- annotateNhoods(milo.myeloid, da_results.myeloid.ctrlibd, coldata_col = "cellstate")
head(da_results.myeloid.ctrlibd)

In [ ]:
celltypeorder.ccduc <- da_results.myeloid.ctrlibd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.myeloid.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colCTRL
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.myeloid.ctrlibd$cellstate)+0.5)

plotDAbeeswarm(da_results.myeloid.ctrlibd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. cCD

In [ ]:
design.myeloid.ccduc <- design.myeloid %>%
    filter(diagSub %in% c("UC", "cCD")) %>%
    droplevels

design.myeloid.ccduc

In [ ]:
da_results.myeloid.ccduc <- testNhoods(milo.myeloid, design = ~version_10x + diagSub, design.df = design.myeloid.ccduc, 
#                                  model.contrasts=c("diagSubUC - diagSubcCD"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.myeloid.ccduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.myeloid.ccduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.myeloid.ccduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.myeloid.ccduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.myeloid, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.ccduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.myeloid.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.ccduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Myeloid Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.myeloid.ccduc.indexed <- da_results.myeloid.ccduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.myeloid))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.myeloid) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.myeloid@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.myeloid.ccduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.myeloid.ccduc <- annotateNhoods(milo.myeloid, da_results.myeloid.ccduc, coldata_col = "cellstate")
head(da_results.myeloid.ccduc)

In [ ]:
celltypeorder.ccduc <- da_results.myeloid.ccduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.myeloid.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.myeloid.ccduc$cellstate)+0.5)

plotDAbeeswarm(da_results.myeloid.ccduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. FGID

In [ ]:
design.myeloid.fgiduc <- design.myeloid %>%
    filter(diagSub %in% c("FGID", "UC")) %>%
    droplevels

design.myeloid.fgiduc

In [ ]:
da_results.myeloid.fgiduc <- testNhoods(milo.myeloid, design = ~version_10x + diagSub, design.df = design.myeloid.fgiduc, 
#                                  model.contrasts=c("diagSubUC - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.myeloid.fgiduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.myeloid.fgiduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.myeloid.fgiduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.myeloid.fgiduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.myeloid, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.fgiduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.myeloid.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.fgiduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Myeloid Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.myeloid.fgiduc.indexed <- da_results.myeloid.fgiduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.myeloid))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.myeloid) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.myeloid@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.myeloid.fgiduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.myeloid.fgiduc <- annotateNhoods(milo.myeloid, da_results.myeloid.fgiduc, coldata_col = "cellstate")
head(da_results.myeloid.fgiduc)

In [ ]:
celltypeorder.fgiduc <- da_results.myeloid.fgiduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.myeloid.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.myeloid.fgiduc$cellstate)+0.5)

plotDAbeeswarm(da_results.myeloid.fgiduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgiduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## cCD vs. FGID

In [ ]:
design.myeloid.fgidccd <- design.myeloid %>%
    filter(diagSub %in% c("FGID", "cCD")) %>%
    droplevels

design.myeloid.fgidccd

In [ ]:
da_results.myeloid.fgidccd <- testNhoods(milo.myeloid, design = ~version_10x + diagSub, design.df = design.myeloid.fgidccd, 
#                                  model.contrasts=c("diagSubcCD - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.myeloid.fgidccd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.myeloid.fgidccd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.myeloid.fgidccd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.myeloid.fgidccd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.myeloid, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.fgidccd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.myeloid.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.myeloid, da_results.myeloid.fgidccd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Myeloid Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.myeloid.fgidccd.indexed <- da_results.myeloid.fgidccd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.myeloid))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.myeloid) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.myeloid@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.myeloid.fgidccd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (cCD vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.myeloid.fgidccd <- annotateNhoods(milo.myeloid, da_results.myeloid.fgidccd, coldata_col = "cellstate")
head(da_results.myeloid.fgidccd)

In [ ]:
celltypeorder.fgidccd <- da_results.myeloid.fgidccd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.myeloid.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.myeloid.fgidccd$cellstate)+0.5)

plotDAbeeswarm(da_results.myeloid.fgidccd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgidccd))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## Saves

### DA Results

In [ ]:
write_csv(da_results.myeloid.ctrlibd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ctrlibd.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.myeloid.ccduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ccduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.myeloid.fgiduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgiduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.myeloid.fgidccd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgidccd.csv", 
                 thiscelltype, thiscelltype)
         )

### Neighbourhoods

In [ ]:
Matrix::writeMM(nhoods(milo.myeloid), 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_sparse.txt", 
                thiscelltype, thiscelltype)
       )

In [ ]:
write_tsv(as.data.frame(as.matrix(nhoodCounts(milo.myeloid))) %>% rownames_to_column("nhood"),
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_dense.txt", 
                thiscelltype, thiscelltype)
       )

### Milo Object

In [ ]:
milo.myeloid

In [ ]:
saveRDS(milo.myeloid, 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x.rds", thiscelltype, thiscelltype)
       )

# Plasma

## Pre-Processing

In [ ]:
thiscelltype <- "plasma"

In [ ]:
sce.plasma <- readH5AD(
    sprintf('../../predict/data/objects/annotated/predictCOL__CB_%s_annotated_231226.h5ad', thiscelltype)
)
sce.plasma

In [ ]:
sce.plasma <- subset(sce.plasma, , timepoint == "T0" & diagSub != "iCD")
sce.plasma

In [ ]:
sce.plasma$diagSub <- droplevels(sce.plasma$diagSub)
sce.plasma$timepoint <- droplevels(sce.plasma$timepoint)
sce.plasma$group <- droplevels(sce.plasma$group)

In [ ]:
alldf.plasma <- data.frame(barcode = colnames(sce.plasma), diagSub = sce.plasma$diagSub)
alldf.plasma

In [ ]:
groupCount.plasma <- alldf.plasma %>% pull(diagSub) %>% table
groupCount.plasma

In [ ]:
dsVal.plasma <- groupCount.plasma %>% min
dsVal.plasma

In [ ]:
set.seed(thisseed)
dsdf.plasma <- alldf.plasma %>%
    group_by(diagSub) %>% 
    sample_n(dsVal.plasma, )

dsdf.plasma

In [ ]:
dsdf.plasma %>% pull(diagSub) %>% table

In [ ]:
# perform downsample
sce.plasma <- subset(sce.plasma, , colnames(sce.plasma) %in% dsdf.plasma$barcode)
sce.plasma

In [ ]:
sce.plasma$biopsy <- droplevels(sce.plasma$biopsy)

In [ ]:
num_cells <- sort(table(sce.plasma$biopsy))
num_cells

In [ ]:
filter_out <- num_cells[num_cells < 5] %>% names
ifelse(identical(filter_out, character(0)), 
       print("No Biopsies Removed"), 
       filter_out)

In [ ]:
# remove biopsies w/ fewer than 5 cells
sce.plasma <- subset(sce.plasma, , biopsy %notin% filter_out)
sce.plasma

In [ ]:
sce.plasma$biopsy <- droplevels(sce.plasma$biopsy)

In [ ]:
sort(table(sce.plasma$biopsy))

In [ ]:
n_distinct(sce.plasma$biopsy)

In [ ]:
milo.plasma <- Milo(sce.plasma)
milo.plasma

In [ ]:
k.plasma <- 200
d.plasma <- 30
prop.plasma <- 0.025

In [ ]:
milo.plasma <- buildGraph(milo.plasma, k = k.plasma, d = d.plasma, reduced.dim = "X_scVI")
milo.plasma

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=120)
milo.plasma <- makeNhoods(milo.plasma, prop = prop.plasma, k = k.plasma, d=d.plasma, refined = TRUE, reduced_dims = "X_scVI")
plotNhoodSizeHist(milo.plasma)

In [ ]:
milo.plasma <- countCells(milo.plasma, meta.data = data.frame(colData(milo.plasma)), samples="biopsy")
head(nhoodCounts(milo.plasma))

In [ ]:
dim(nhoodCounts(milo.plasma))

In [ ]:
nhoodCounts(milo.plasma) %>% rowSums %>% range

In [ ]:
nhoodCounts(milo.plasma) %>% rowSums %>% range %>% .[1]/dim(nhoodCounts(milo.plasma))[2]

In [ ]:
nhoodCounts(milo.plasma) %>% rowSums %>% mean

In [ ]:
nhoodCounts(milo.plasma) %>% rowSums %>% median

In [ ]:
dim(nhoodCounts(milo.plasma))[2]*5

In [ ]:
nhoodCounts(milo.plasma) %>% rowSums %>% mean > dim(nhoodCounts(milo.plasma))[2]*5

In [ ]:
nhoodCounts(milo.plasma) %>% colSums %>% sort

In [ ]:
milo.plasma <- calcNhoodDistance(milo.plasma, d=d.plasma, reduced.dim = "X_scVI")

In [ ]:
milo.plasma <- buildNhoodGraph(milo.plasma)

## Design

In [ ]:
diagSub.vector <- colData(milo.plasma) %>%
    as.tibble %>%
    mutate(diagSub = fct_relevel(diagSub, c("FGID", "iCD", "cCD", "UC"))) %>%
    pull(diagSub)

In [ ]:
group.vector <- colData(milo.plasma) %>%
    as.tibble %>%
    mutate(group = fct_relevel(group, c("CTRL", "ILEAL", "IBD"))) %>%
    pull(group)

In [ ]:
biopsy.vector <- colData(milo.plasma) %>%
    as.tibble %>%
    mutate(biopsy = as.factor(biopsy)) %>%
    pull(biopsy)

In [ ]:
version_10x.vector <- colData(milo.plasma) %>%
    as.tibble %>%
    mutate(version_10x = as.factor(version_10x)) %>%
    pull(version_10x)

In [ ]:
milo.plasma$diagSub <- diagSub.vector
milo.plasma$group <- group.vector
milo.plasma$biopsy <- biopsy.vector
milo.plasma$version_10x <- version_10x.vector

In [ ]:
design.plasma <- data.frame(colData(milo.plasma))[,c("biopsy", "diagSub", "group", "version_10x")]
design.plasma <- distinct(design.plasma)

rownames(design.plasma) <- design.plasma$biopsy

## Reorder rownames to match columns of nhoodCounts(milo)
design.plasma <- design.plasma[colnames(nhoodCounts(milo.plasma)), , drop=FALSE]

head(design.plasma)
levels(design.plasma$diagSub)
levels(design.plasma$group)
levels(design.plasma$version_10x)

## Prepare Seurat Objects

This is for plotting purposes

In [ ]:
seuobj.plasma <- as.Seurat(milo.plasma, counts = "counts", data = NULL)
seuobj.plasma

In [ ]:
seuobj.plasma <- AddMetaData(seuobj.plasma, seuobj.plasma@reductions$X_umap@cell.embeddings %>% as.data.frame)
seuobj.plasma@meta.data

## Set Plot Limits

In [ ]:
topX <- 8.5
topY <- 7.75

bottomX <- 8.5
bottomY <- 0

midX <- 9.75
midY <- 4
midLabel <- "**LogFC**"

limsX <- c(-4.5,7)
limsY <- c(0, 8)

## IBD vs. CTRL

In [ ]:
design.plasma.ctrlibd <- design.plasma %>%
    filter(group %in% c("CTRL", "IBD")) %>%
    droplevels

design.plasma.ctrlibd

In [ ]:
da_results.plasma.ctrlibd <- testNhoods(milo.plasma, design = ~version_10x + group, design.df = design.plasma.ctrlibd, 
#                                  model.contrasts=c("groupIBD - groupCTRL"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.plasma.ctrlibd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.plasma.ctrlibd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.plasma.ctrlibd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.plasma.ctrlibd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.plasma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.plasma, da_results.plasma.ctrlibd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.plasma.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.plasma, da_results.plasma.ctrlibd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Plasma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.ctrlibd.indexed <- da_results.plasma.ctrlibd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.plasma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.plasma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.plasma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.plasma.ctrlibd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.ctrlibd <- annotateNhoods(milo.plasma, da_results.plasma.ctrlibd, coldata_col = "cellstate")
head(da_results.plasma.ctrlibd)

In [ ]:
celltypeorder.ccduc <- da_results.plasma.ctrlibd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.plasma.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colCTRL
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.plasma.ctrlibd$cellstate)+0.5)

plotDAbeeswarm(da_results.plasma.ctrlibd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. cCD

In [ ]:
design.plasma.ccduc <- design.plasma %>%
    filter(diagSub %in% c("UC", "cCD")) %>%
    droplevels

design.plasma.ccduc

In [ ]:
da_results.plasma.ccduc <- testNhoods(milo.plasma, design = ~version_10x + diagSub, design.df = design.plasma.ccduc, 
#                                  model.contrasts=c("diagSubUC - diagSubcCD"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.plasma.ccduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.plasma.ccduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.plasma.ccduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.plasma.ccduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.plasma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.plasma, da_results.plasma.ccduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.plasma.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD

plotNhoodGraphDA(milo.plasma, da_results.plasma.ccduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Plasma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.ccduc.indexed <- da_results.plasma.ccduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.plasma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.plasma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.plasma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.plasma.ccduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. cCD)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.ccduc <- annotateNhoods(milo.plasma, da_results.plasma.ccduc, coldata_col = "cellstate")
head(da_results.plasma.ccduc)

In [ ]:
celltypeorder.ccduc <- da_results.plasma.ccduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.plasma.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.plasma.ccduc$cellstate)+0.5)

plotDAbeeswarm(da_results.plasma.ccduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. FGID

In [ ]:
design.plasma.fgiduc <- design.plasma %>%
    filter(diagSub %in% c("FGID", "UC")) %>%
    droplevels

design.plasma.fgiduc

In [ ]:
da_results.plasma.fgiduc <- testNhoods(milo.plasma, design = ~version_10x + diagSub, design.df = design.plasma.fgiduc, 
#                                  model.contrasts=c("diagSubUC - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.plasma.fgiduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.plasma.fgiduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.plasma.fgiduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.plasma.fgiduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.plasma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.plasma, da_results.plasma.fgiduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.plasma.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.plasma, da_results.plasma.fgiduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Plasma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.fgiduc.indexed <- da_results.plasma.fgiduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.plasma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.plasma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.plasma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.plasma.fgiduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (UC vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.fgiduc <- annotateNhoods(milo.plasma, da_results.plasma.fgiduc, coldata_col = "cellstate")
head(da_results.plasma.fgiduc)

In [ ]:
celltypeorder.fgiduc <- da_results.plasma.fgiduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.plasma.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.plasma.fgiduc$cellstate)+0.5)

plotDAbeeswarm(da_results.plasma.fgiduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgiduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## cCD vs. FGID

In [ ]:
design.plasma.fgidccd <- design.plasma %>%
    filter(diagSub %in% c("FGID", "cCD")) %>%
    droplevels

design.plasma.fgidccd

In [ ]:
da_results.plasma.fgidccd <- testNhoods(milo.plasma, design = ~version_10x + diagSub, design.df = design.plasma.fgidccd, 
#                                  model.contrasts=c("diagSubcCD - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.plasma.fgidccd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.plasma.fgidccd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.plasma.fgidccd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.plasma.fgidccd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.plasma, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.plasma, da_results.plasma.fgidccd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.plasma.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.plasma, da_results.plasma.fgidccd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "Plasma Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.fgidccd.indexed <- da_results.plasma.fgidccd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.plasma))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.plasma) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.plasma@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.plasma.fgidccd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
  labs(fill = "LogFC", title = sprintf("%s Diff. Abundance (cCD vs. FGID)", str_to_title(thiscelltype))) + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.plasma.fgidccd <- annotateNhoods(milo.plasma, da_results.plasma.fgidccd, coldata_col = "cellstate")
head(da_results.plasma.fgidccd)

In [ ]:
celltypeorder.fgidccd <- da_results.plasma.fgidccd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.plasma.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.plasma.fgidccd$cellstate)+0.5)

plotDAbeeswarm(da_results.plasma.fgidccd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgidccd))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## Saves

### DA Results

In [ ]:
write_csv(da_results.plasma.ctrlibd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ctrlibd.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.plasma.ccduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ccduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.plasma.fgiduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgiduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.plasma.fgidccd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgidccd.csv", 
                 thiscelltype, thiscelltype)
         )

### Neighbourhoods

In [ ]:
Matrix::writeMM(nhoods(milo.plasma), 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_sparse.txt", 
                thiscelltype, thiscelltype)
       )

In [ ]:
write_tsv(as.data.frame(as.matrix(nhoodCounts(milo.plasma))) %>% rownames_to_column("nhood"),
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_dense.txt", 
                thiscelltype, thiscelltype)
       )

### Milo Object

In [ ]:
milo.plasma

In [ ]:
saveRDS(milo.plasma, 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x.rds", thiscelltype, thiscelltype)
       )

# BCell

## Pre-Processing

In [ ]:
thiscelltype <- "bcell"

In [ ]:
sce.bcell <- readH5AD(
    sprintf('../../predict/data/objects/annotated/predictCOL__CB_%s_annotated_231226.h5ad', thiscelltype)
)
sce.bcell

In [ ]:
sce.bcell <- subset(sce.bcell, , timepoint == "T0" & diagSub != "iCD")
sce.bcell

In [ ]:
sce.bcell$diagSub <- droplevels(sce.bcell$diagSub)
sce.bcell$timepoint <- droplevels(sce.bcell$timepoint)
sce.bcell$group <- droplevels(sce.bcell$group)

In [ ]:
alldf.bcell <- data.frame(barcode = colnames(sce.bcell), diagSub = sce.bcell$diagSub)
alldf.bcell

In [ ]:
groupCount.bcell <- alldf.bcell %>% pull(diagSub) %>% table
groupCount.bcell

In [ ]:
dsVal.bcell <- groupCount.bcell %>% min
dsVal.bcell

In [ ]:
set.seed(thisseed)
dsdf.bcell <- alldf.bcell %>%
    group_by(diagSub) %>% 
    sample_n(dsVal.bcell, )

dsdf.bcell

In [ ]:
dsdf.bcell %>% pull(diagSub) %>% table

In [ ]:
# perform downsample
sce.bcell <- subset(sce.bcell, , colnames(sce.bcell) %in% dsdf.bcell$barcode)
sce.bcell

In [ ]:
sce.bcell$biopsy <- droplevels(sce.bcell$biopsy)

In [ ]:
num_cells <- sort(table(sce.bcell$biopsy))
num_cells

In [ ]:
filter_out <- num_cells[num_cells < 5] %>% names
ifelse(identical(filter_out, character(0)), 
       print("No Biopsies Removed"), 
       filter_out)

In [ ]:
# remove biopsies w/ fewer than 5 cells
sce.bcell <- subset(sce.bcell, , biopsy %notin% filter_out)
sce.bcell

In [ ]:
sce.bcell$biopsy <- droplevels(sce.bcell$biopsy)

In [ ]:
sort(table(sce.bcell$biopsy))

In [ ]:
n_distinct(sce.bcell$biopsy)

In [ ]:
milo.bcell <- Milo(sce.bcell)
milo.bcell

In [ ]:
k.bcell <- 200
d.bcell <- 30
prop.bcell <- 0.05

In [ ]:
milo.bcell <- buildGraph(milo.bcell, k = k.bcell, d = d.bcell, reduced.dim = "X_scVI")
milo.bcell

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=120)
milo.bcell <- makeNhoods(milo.bcell, prop = prop.bcell, k = k.bcell, d=d.bcell, refined = TRUE, reduced_dims = "X_scVI")
plotNhoodSizeHist(milo.bcell)

In [ ]:
milo.bcell <- countCells(milo.bcell, meta.data = data.frame(colData(milo.bcell)), samples="biopsy")
head(nhoodCounts(milo.bcell))

In [ ]:
dim(nhoodCounts(milo.bcell))

In [ ]:
nhoodCounts(milo.bcell) %>% rowSums %>% range

In [ ]:
nhoodCounts(milo.bcell) %>% rowSums %>% range %>% .[1]/dim(nhoodCounts(milo.bcell))[2]

In [ ]:
nhoodCounts(milo.bcell) %>% rowSums %>% mean

In [ ]:
nhoodCounts(milo.bcell) %>% rowSums %>% median

In [ ]:
dim(nhoodCounts(milo.bcell))[2]*5

In [ ]:
nhoodCounts(milo.bcell) %>% rowSums %>% mean > dim(nhoodCounts(milo.bcell))[2]*5

In [ ]:
nhoodCounts(milo.bcell) %>% colSums %>% sort

In [ ]:
milo.bcell <- calcNhoodDistance(milo.bcell, d=d.bcell, reduced.dim = "X_scVI")

In [ ]:
milo.bcell <- buildNhoodGraph(milo.bcell)

## Design

In [ ]:
diagSub.vector <- colData(milo.bcell) %>%
    as.tibble %>%
    mutate(diagSub = fct_relevel(diagSub, c("FGID", "iCD", "cCD", "UC"))) %>%
    pull(diagSub)

In [ ]:
group.vector <- colData(milo.bcell) %>%
    as.tibble %>%
    mutate(group = fct_relevel(group, c("CTRL", "ILEAL", "IBD"))) %>%
    pull(group)

In [ ]:
biopsy.vector <- colData(milo.bcell) %>%
    as.tibble %>%
    mutate(biopsy = as.factor(biopsy)) %>%
    pull(biopsy)

In [ ]:
version_10x.vector <- colData(milo.bcell) %>%
    as.tibble %>%
    mutate(version_10x = as.factor(version_10x)) %>%
    pull(version_10x)

In [ ]:
milo.bcell$diagSub <- diagSub.vector
milo.bcell$group <- group.vector
milo.bcell$biopsy <- biopsy.vector
milo.bcell$version_10x <- version_10x.vector

In [ ]:
design.bcell <- data.frame(colData(milo.bcell))[,c("biopsy", "diagSub", "group", "version_10x")]
design.bcell <- distinct(design.bcell)

rownames(design.bcell) <- design.bcell$biopsy

## Reorder rownames to match columns of nhoodCounts(milo)
design.bcell <- design.bcell[colnames(nhoodCounts(milo.bcell)), , drop=FALSE]

head(design.bcell)
levels(design.bcell$diagSub)
levels(design.bcell$group)
levels(design.bcell$version_10x)

## Prepare Seurat Objects

This is for plotting purposes

In [ ]:
seuobj.bcell <- as.Seurat(milo.bcell, counts = "counts", data = NULL)
seuobj.bcell

In [ ]:
seuobj.bcell <- AddMetaData(seuobj.bcell, seuobj.bcell@reductions$X_umap@cell.embeddings %>% as.data.frame)
seuobj.bcell@meta.data

## Set Plot Limits

In [ ]:
topX <- 13.5
topY <- 12.25

bottomX <- 13.5
bottomY <- 0

midX <- 15.5
midY <- 6
midLabel <- "**LogFC**"

limsX <- c(-5,11.5)
limsY <- c(0, 12.5)

## IBD vs. CTRL

In [ ]:
design.bcell.ctrlibd <- design.bcell %>%
    filter(group %in% c("CTRL", "IBD")) %>%
    droplevels

design.bcell.ctrlibd

In [ ]:
da_results.bcell.ctrlibd <- testNhoods(milo.bcell, design = ~version_10x + group, design.df = design.bcell.ctrlibd, 
#                                  model.contrasts=c("groupIBD - groupCTRL"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.bcell.ctrlibd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.bcell.ctrlibd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.bcell.ctrlibd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.bcell.ctrlibd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.bcell, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.bcell, da_results.bcell.ctrlibd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.bcell.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.bcell, da_results.bcell.ctrlibd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
thiscelltype

In [ ]:
da_results.bcell.ctrlibd.indexed <- da_results.bcell.ctrlibd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.bcell))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.bcell) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.bcell@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.bcell.ctrlibd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.bcell.ctrlibd <- annotateNhoods(milo.bcell, da_results.bcell.ctrlibd, coldata_col = "cellstate")
head(da_results.bcell.ctrlibd)

In [ ]:
celltypeorder.ccduc <- da_results.bcell.ctrlibd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.bcell.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colCTRL
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.bcell.ctrlibd$cellstate)+0.5)

plotDAbeeswarm(da_results.bcell.ctrlibd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. cCD

In [ ]:
design.bcell.ccduc <- design.bcell %>%
    filter(diagSub %in% c("UC", "cCD")) %>%
    droplevels

design.bcell.ccduc

In [ ]:
da_results.bcell.ccduc <- testNhoods(milo.bcell, design = ~version_10x + diagSub, design.df = design.bcell.ccduc, 
#                                  model.contrasts=c("diagSubUC - diagSubcCD"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.bcell.ccduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.bcell.ccduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.bcell.ccduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.bcell.ccduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.bcell, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.bcell, da_results.bcell.ccduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.bcell.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD

plotNhoodGraphDA(milo.bcell, da_results.bcell.ccduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.bcell.ccduc.indexed <- da_results.bcell.ccduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.bcell))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.bcell) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.bcell@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.bcell.ccduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.bcell.ccduc <- annotateNhoods(milo.bcell, da_results.bcell.ccduc, coldata_col = "cellstate")
head(da_results.bcell.ccduc)

In [ ]:
celltypeorder.ccduc <- da_results.bcell.ccduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.bcell.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.bcell.ccduc$cellstate)+0.5)

plotDAbeeswarm(da_results.bcell.ccduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. FGID

In [ ]:
design.bcell.fgiduc <- design.bcell %>%
    filter(diagSub %in% c("FGID", "UC")) %>%
    droplevels

design.bcell.fgiduc

In [ ]:
da_results.bcell.fgiduc <- testNhoods(milo.bcell, design = ~version_10x + diagSub, design.df = design.bcell.fgiduc, 
#                                  model.contrasts=c("diagSubUC - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.bcell.fgiduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.bcell.fgiduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.bcell.fgiduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.bcell.fgiduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.bcell, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.bcell, da_results.bcell.fgiduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.bcell.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.bcell, da_results.bcell.fgiduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.bcell.fgiduc.indexed <- da_results.bcell.fgiduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.bcell))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.bcell) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.bcell@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.bcell.fgiduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.bcell.fgiduc <- annotateNhoods(milo.bcell, da_results.bcell.fgiduc, coldata_col = "cellstate")
head(da_results.bcell.fgiduc)

In [ ]:
celltypeorder.fgiduc <- da_results.bcell.fgiduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.bcell.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.bcell.fgiduc$cellstate)+0.5)

plotDAbeeswarm(da_results.bcell.fgiduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgiduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## cCD vs. FGID

In [ ]:
design.bcell.fgidccd <- design.bcell %>%
    filter(diagSub %in% c("FGID", "cCD")) %>%
    droplevels

design.bcell.fgidccd

In [ ]:
da_results.bcell.fgidccd <- testNhoods(milo.bcell, design = ~version_10x + diagSub, design.df = design.bcell.fgidccd, 
#                                  model.contrasts=c("diagSubcCD - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.bcell.fgidccd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.bcell.fgidccd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.bcell.fgidccd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.bcell.fgidccd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.bcell, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.bcell, da_results.bcell.fgidccd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.bcell.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.bcell, da_results.bcell.fgidccd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.bcell.fgidccd.indexed <- da_results.bcell.fgidccd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.bcell))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.bcell) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.bcell@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.bcell.fgidccd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "B Cell Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.bcell.fgidccd <- annotateNhoods(milo.bcell, da_results.bcell.fgidccd, coldata_col = "cellstate")
head(da_results.bcell.fgidccd)

In [ ]:
celltypeorder.fgidccd <- da_results.bcell.fgidccd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.bcell.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.bcell.fgidccd$cellstate)+0.5)

plotDAbeeswarm(da_results.bcell.fgidccd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgidccd))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## Saves

### DA Results

In [ ]:
write_csv(da_results.bcell.ctrlibd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ctrlibd.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.bcell.ccduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ccduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.bcell.fgiduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgiduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.bcell.fgidccd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgidccd.csv", 
                 thiscelltype, thiscelltype)
         )

### Neighbourhoods

In [ ]:
Matrix::writeMM(nhoods(milo.bcell), 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_sparse.txt", 
                thiscelltype, thiscelltype)
       )

In [ ]:
write_tsv(as.data.frame(as.matrix(nhoodCounts(milo.bcell))) %>% rownames_to_column("nhood"),
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_dense.txt", 
                thiscelltype, thiscelltype)
       )

### Milo Object

In [ ]:
milo.bcell

In [ ]:
saveRDS(milo.bcell, 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x.rds", thiscelltype, thiscelltype)
       )

In [ ]:
rm(sce.bcell)
gc()

# TNKILC

## Pre-Processing

In [ ]:
thiscelltype <- "tnkilc"

In [ ]:
sce.tnkilc <- readH5AD(
    sprintf('../../predict/data/objects/annotated/predictCOL__CB_%s_annotated_231226.h5ad', thiscelltype)
)
sce.tnkilc

In [ ]:
sce.tnkilc <- subset(sce.tnkilc, , timepoint == "T0" & diagSub != "iCD")
sce.tnkilc

In [ ]:
sce.tnkilc$diagSub <- droplevels(sce.tnkilc$diagSub)
sce.tnkilc$timepoint <- droplevels(sce.tnkilc$timepoint)
sce.tnkilc$group <- droplevels(sce.tnkilc$group)

In [ ]:
alldf.tnkilc <- data.frame(barcode = colnames(sce.tnkilc), diagSub = sce.tnkilc$diagSub)
alldf.tnkilc

In [ ]:
groupCount.tnkilc <- alldf.tnkilc %>% pull(diagSub) %>% table
groupCount.tnkilc

In [ ]:
dsVal.tnkilc <- groupCount.tnkilc %>% min
dsVal.tnkilc

In [ ]:
set.seed(thisseed)
dsdf.tnkilc <- alldf.tnkilc %>%
    group_by(diagSub) %>% 
    sample_n(dsVal.tnkilc, )

dsdf.tnkilc

In [ ]:
dsdf.tnkilc %>% pull(diagSub) %>% table

In [ ]:
# perform downsample
sce.tnkilc <- subset(sce.tnkilc, , colnames(sce.tnkilc) %in% dsdf.tnkilc$barcode)
sce.tnkilc

In [ ]:
sce.tnkilc$biopsy <- droplevels(sce.tnkilc$biopsy)

In [ ]:
num_cells <- sort(table(sce.tnkilc$biopsy))
num_cells

In [ ]:
filter_out <- num_cells[num_cells < 5] %>% names
ifelse(identical(filter_out, character(0)), 
       print("No Biopsies Removed"), 
       filter_out)

In [ ]:
# remove biopsies w/ fewer than 5 cells
sce.tnkilc <- subset(sce.tnkilc, , biopsy %notin% filter_out)
sce.tnkilc

In [ ]:
sce.tnkilc$biopsy <- droplevels(sce.tnkilc$biopsy)

In [ ]:
sort(table(sce.tnkilc$biopsy))

In [ ]:
n_distinct(sce.tnkilc$biopsy)

In [ ]:
milo.tnkilc <- Milo(sce.tnkilc)
milo.tnkilc

In [ ]:
rm(sce.tnkilc)
gc()

In [ ]:
k.tnkilc <- 200
d.tnkilc <- 30
prop.tnkilc <- 0.025

In [ ]:
milo.tnkilc <- buildGraph(milo.tnkilc, k = k.tnkilc, d = d.tnkilc, reduced.dim = "X_scVI")
milo.tnkilc

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=120)
milo.tnkilc <- makeNhoods(milo.tnkilc, prop = prop.tnkilc, k = k.tnkilc, d=d.tnkilc, refined = TRUE, reduced_dims = "X_scVI")
plotNhoodSizeHist(milo.tnkilc)

In [ ]:
milo.tnkilc <- countCells(milo.tnkilc, meta.data = data.frame(colData(milo.tnkilc)), samples="biopsy")
head(nhoodCounts(milo.tnkilc))

In [ ]:
dim(nhoodCounts(milo.tnkilc))

In [ ]:
nhoodCounts(milo.tnkilc) %>% rowSums %>% range

In [ ]:
nhoodCounts(milo.tnkilc) %>% rowSums %>% range %>% .[1]/dim(nhoodCounts(milo.tnkilc))[2]

In [ ]:
nhoodCounts(milo.tnkilc) %>% rowSums %>% mean

In [ ]:
nhoodCounts(milo.tnkilc) %>% rowSums %>% median

In [ ]:
dim(nhoodCounts(milo.tnkilc))[2]*5

In [ ]:
nhoodCounts(milo.tnkilc) %>% rowSums %>% mean > dim(nhoodCounts(milo.tnkilc))[2]*5

In [ ]:
nhoodCounts(milo.tnkilc) %>% colSums %>% sort

In [ ]:
milo.tnkilc <- calcNhoodDistance(milo.tnkilc, d=d.tnkilc, reduced.dim = "X_scVI")

In [ ]:
milo.tnkilc <- buildNhoodGraph(milo.tnkilc)

## Design

In [ ]:
diagSub.vector <- colData(milo.tnkilc) %>%
    as.tibble %>%
    mutate(diagSub = fct_relevel(diagSub, c("FGID", "iCD", "cCD", "UC"))) %>%
    pull(diagSub)

In [ ]:
group.vector <- colData(milo.tnkilc) %>%
    as.tibble %>%
    mutate(group = fct_relevel(group, c("CTRL", "ILEAL", "IBD"))) %>%
    pull(group)

In [ ]:
biopsy.vector <- colData(milo.tnkilc) %>%
    as.tibble %>%
    mutate(biopsy = as.factor(biopsy)) %>%
    pull(biopsy)

In [ ]:
version_10x.vector <- colData(milo.tnkilc) %>%
    as.tibble %>%
    mutate(version_10x = as.factor(version_10x)) %>%
    pull(version_10x)

In [ ]:
milo.tnkilc$diagSub <- diagSub.vector
milo.tnkilc$group <- group.vector
milo.tnkilc$biopsy <- biopsy.vector
milo.tnkilc$version_10x <- version_10x.vector

In [ ]:
design.tnkilc <- data.frame(colData(milo.tnkilc))[,c("biopsy", "diagSub", "group", "version_10x")]
design.tnkilc <- distinct(design.tnkilc)

rownames(design.tnkilc) <- design.tnkilc$biopsy

## Reorder rownames to match columns of nhoodCounts(milo)
design.tnkilc <- design.tnkilc[colnames(nhoodCounts(milo.tnkilc)), , drop=FALSE]

head(design.tnkilc)
levels(design.tnkilc$diagSub)
levels(design.tnkilc$group)
levels(design.tnkilc$version_10x)

## Prepare Seurat Objects

This is for plotting purposes

In [ ]:
seuobj.tnkilc <- as.Seurat(milo.tnkilc, counts = "counts", data = NULL)
seuobj.tnkilc

In [ ]:
seuobj.tnkilc <- AddMetaData(seuobj.tnkilc, seuobj.tnkilc@reductions$X_umap@cell.embeddings %>% as.data.frame)
seuobj.tnkilc@meta.data

## Set Plot Limits

In [ ]:
topX <- 15.5
topY <- 9.25

bottomX <- 15.5
bottomY <- 1

midX <- 16.5
midY <- 5
midLabel <- "**LogFC**"

limsX <- c(3,14)
limsY <- c(1, 9.5)

## IBD vs. CTRL

In [ ]:
design.tnkilc.ctrlibd <- design.tnkilc %>%
    filter(group %in% c("CTRL", "IBD")) %>%
    droplevels

design.tnkilc.ctrlibd

In [ ]:
da_results.tnkilc.ctrlibd <- testNhoods(milo.tnkilc, design = ~version_10x + group, design.df = design.tnkilc.ctrlibd, 
#                                  model.contrasts=c("groupIBD - groupCTRL"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.tnkilc.ctrlibd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.tnkilc.ctrlibd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.tnkilc.ctrlibd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.tnkilc.ctrlibd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.tnkilc, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.ctrlibd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.tnkilc.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.ctrlibd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
thiscelltype

In [ ]:
da_results.tnkilc.ctrlibd.indexed <- da_results.tnkilc.ctrlibd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.tnkilc))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.tnkilc) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.tnkilc@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.tnkilc.ctrlibd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.tnkilc.ctrlibd <- annotateNhoods(milo.tnkilc, da_results.tnkilc.ctrlibd, coldata_col = "cellstate")
head(da_results.tnkilc.ctrlibd)

In [ ]:
celltypeorder.ccduc <- da_results.tnkilc.ctrlibd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.tnkilc.ctrlibd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**IBD**"
topColour <- colIBD
bottomLabel <- "**CTRL**"
bottomColour <- colCTRL
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.tnkilc.ctrlibd$cellstate)+0.5)

plotDAbeeswarm(da_results.tnkilc.ctrlibd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. cCD

In [ ]:
design.tnkilc.ccduc <- design.tnkilc %>%
    filter(diagSub %in% c("UC", "cCD")) %>%
    droplevels

design.tnkilc.ccduc

In [ ]:
da_results.tnkilc.ccduc <- testNhoods(milo.tnkilc, design = ~version_10x + diagSub, design.df = design.tnkilc.ccduc, 
#                                  model.contrasts=c("diagSubUC - diagSubcCD"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.tnkilc.ccduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.tnkilc.ccduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.tnkilc.ccduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.tnkilc.ccduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.tnkilc, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.ccduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.tnkilc.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.ccduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.tnkilc.ccduc.indexed <- da_results.tnkilc.ccduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.tnkilc))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.tnkilc) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.tnkilc@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.tnkilc.ccduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.tnkilc.ccduc <- annotateNhoods(milo.tnkilc, da_results.tnkilc.ccduc, coldata_col = "cellstate")
head(da_results.tnkilc.ccduc)

In [ ]:
celltypeorder.ccduc <- da_results.tnkilc.ccduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.tnkilc.ccduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**cCD**"
bottomColour <- colcCD
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.tnkilc.ccduc$cellstate)+0.5)

plotDAbeeswarm(da_results.tnkilc.ccduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.ccduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## UC vs. FGID

In [ ]:
design.tnkilc.fgiduc <- design.tnkilc %>%
    filter(diagSub %in% c("FGID", "UC")) %>%
    droplevels

design.tnkilc.fgiduc

In [ ]:
da_results.tnkilc.fgiduc <- testNhoods(milo.tnkilc, design = ~version_10x + diagSub, design.df = design.tnkilc.fgiduc, 
#                                  model.contrasts=c("diagSubUC - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.tnkilc.fgiduc %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.tnkilc.fgiduc, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.tnkilc.fgiduc, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.tnkilc.fgiduc, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.tnkilc, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.fgiduc, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.tnkilc.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.fgiduc, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.tnkilc.fgiduc.indexed <- da_results.tnkilc.fgiduc %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.tnkilc))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.tnkilc) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.tnkilc@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.tnkilc.fgiduc.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.tnkilc.fgiduc <- annotateNhoods(milo.tnkilc, da_results.tnkilc.fgiduc, coldata_col = "cellstate")
head(da_results.tnkilc.fgiduc)

In [ ]:
celltypeorder.fgiduc <- da_results.tnkilc.fgiduc %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.tnkilc.fgiduc$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**UC**"
topColour <- colUC
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.tnkilc.fgiduc$cellstate)+0.5)

plotDAbeeswarm(da_results.tnkilc.fgiduc %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgiduc))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## cCD vs. FGID

In [ ]:
design.tnkilc.fgidccd <- design.tnkilc %>%
    filter(diagSub %in% c("FGID", "cCD")) %>%
    droplevels

design.tnkilc.fgidccd

In [ ]:
da_results.tnkilc.fgidccd <- testNhoods(milo.tnkilc, design = ~version_10x + diagSub, design.df = design.tnkilc.fgidccd, 
#                                  model.contrasts=c("diagSubcCD - diagSubFGID"), 
                                 reduced.dim = "X_scVI")

In [ ]:
da_results.tnkilc.fgidccd %>%
  arrange(SpatialFDR) %>%
  head() 

options(repr.plot.width=10, repr.plot.height=5, repr.plot.res=120)

ggplot(da_results.tnkilc.fgidccd, aes(PValue)) + geom_histogram(bins=50) +
ggplot(da_results.tnkilc.fgidccd, aes(logFC, -log10(SpatialFDR))) + 
  geom_point() +
  geom_hline(yintercept = 1, color="cyan") + ## Mark significance threshold (10% FDR)
  geom_hline(yintercept = 2, color="blue") ## Mark significance threshold (1% FDR)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=120)
plotNhoodMA(da_results.tnkilc.fgidccd, alpha = alphaVal)

In [ ]:
options(repr.plot.width=12, repr.plot.height=7, repr.plot.res=200)

plotReducedDim(milo.tnkilc, dimred = "X_umap", colour_by="cellstate", text_by = "cellstate", 
                          text_size = 3, point_size=0.5)  + scale_color_discrete() +

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.fgidccd, layout="X_umap",alpha=alphaVal) 

In [ ]:
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)

plotRange <- range(da_results.tnkilc.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID

plotNhoodGraphDA(milo.tnkilc, da_results.tnkilc.fgidccd, layout="X_umap",
                 alpha=alphaVal, node_stroke=0.3, size_range=c(1,4)) + 
    scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                         limits = c(plotMin,plotMax),
                         breaks = seq(plotMin, plotMax, 1)
                        ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
    guides(size = "none", edge_width = "none") +
    theme(
        panel.border = element_rect(colour = "black", fill=NA, size=0.5),
        legend.title = element_blank(),
        legend.position = "right",      
        legend.justification = "center",
        legend.key.height = unit(1.75,'cm'),
        legend.text = element_text(size = 14, face = "bold"),
        legend.background = element_rect(fill = NA),
        plot.title = element_text(hjust=0.5, face = "bold"),
        plot.margin = unit(c(1,3,1,1), "lines")
    ) + 
    coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
    annotate(geom = "richtext", x = topX, y = topY, 
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = bottomX, y = bottomY, 
             label = bottomLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = midX, y = midY, 
             label = midLabel, size = 6, angle=-90, label.colour = NA) +
    ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.tnkilc.fgidccd.indexed <- da_results.tnkilc.fgidccd %>% 
  mutate(NhoodIndexed = unlist(nhoodIndex(milo.tnkilc))[Nhood], .after = Nhood) %>%
  mutate(NhoodSize = nhoodCounts(milo.tnkilc) %>% rowSums, .after = NhoodIndexed) %>% 
  arrange(abs(logFC))

seuobj.tnkilc@meta.data %>% 
  mutate(rownum = row_number()) %>% 
  left_join(da_results.tnkilc.fgidccd.indexed, 
            by = c("rownum" = "NhoodIndexed"), keep = T) %>% 
  filter(rownum %in% NhoodIndexed) %>% 
  mutate(logFC = ifelse(SpatialFDR > alphaVal, 0, logFC)) %>% 
  arrange(abs(logFC)) %>%
  ggplot(aes(x = Xumap_1, y = Xumap_2)) + 
  geom_point(shape = 21, aes(fill = logFC, size = NhoodSize)) +  
  scale_fill_gradient2(low=bottomColour, high=topColour, mid = "white", 
                       limits = c(plotMin,plotMax),
                       breaks = seq(plotMin, plotMax, 1)
  ) + 
    labs(fill = "LogFC", title = "TNKILC Differential Abundance") + 
  guides(size = "none", edge_width = "none") +
  theme(
    panel.border = element_rect(colour = "black", fill=NA, size=0.5),
    legend.title = element_blank(),
    legend.position = "right",      
    legend.justification = "center",
    legend.key.height = unit(1.75,'cm'),
    legend.text = element_text(size = 14, face = "bold"),
    legend.background = element_rect(fill = NA),
    plot.title = element_text(hjust=0.05, face = "bold"),
    plot.margin = unit(c(1,3,1,1), "lines"),
    panel.background = element_rect(fill = NA),
    axis.line = element_blank(), axis.text = element_blank(),
            axis.ticks = element_blank(), axis.title = element_blank()
  ) + 
  coord_cartesian(xlim = limsX, ylim = limsY, clip = "off") +
  annotate(geom = "richtext", x = topX, y = topY, 
           label = topLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = bottomX, y = bottomY, 
           label = bottomLabel, size = 6, label.colour = NA) +
  annotate(geom = "richtext", x = midX, y = midY, 
           label = midLabel, size = 6, angle=-90, label.colour = NA) +
  ggraph::scale_edge_width(range = c(0,0))

In [ ]:
da_results.tnkilc.fgidccd <- annotateNhoods(milo.tnkilc, da_results.tnkilc.fgidccd, coldata_col = "cellstate")
head(da_results.tnkilc.fgidccd)

In [ ]:
celltypeorder.fgidccd <- da_results.tnkilc.fgidccd %>% 
    group_by(cellstate) %>%
    mutate(logFC.mean = mean(logFC, na.rm = T)) %>%
    ungroup %>%
    distinct(cellstate, .keep_all = T) %>%
    arrange(-logFC.mean) %>%
    pull(logFC.mean, cellstate)

plotRange <- range(da_results.tnkilc.fgidccd$logFC, na.rm = T)
plotMin <- floor(plotRange[1])
plotMax <- ceiling(plotRange[2])

topLabel <- "**cCD**"
topColour <- colcCD
bottomLabel <- "**FGID**"
bottomColour <- colFGID
midLabel <- "**LogFC**"

topX.bee <- plotMax
bottomX.bee <- plotMin
limY.bee <- c(0.5,n_distinct(da_results.tnkilc.fgidccd$cellstate)+0.5)

plotDAbeeswarm(da_results.tnkilc.fgidccd %>% 
                 mutate(cellstate = fct_relevel(cellstate, names(celltypeorder.fgidccd))), 
               group.by = "cellstate", alpha = alphaVal) +
    geom_hline(yintercept = 0, alpha = 0.5) +
    labs(x = NULL) +
    # scale_x_discrete(limits = rev) +
    scale_y_continuous(limits = c(plotMin,plotMax), breaks = seq(plotMin, plotMax, 2)) +
    scale_color_gradient2(low = bottomColour, mid = "gray90", high = topColour, na.value = "gray75") +
    coord_flip(xlim = limY.bee, ylim = c(plotMin, plotMax), clip = "off") +
    theme(
        plot.margin = unit(c(1,0.25,0.25,0.25), "cm"),
        axis.text.y = element_text(face = "bold"),
        panel.grid.minor.y = element_blank()
    ) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = topX.bee,
             label = topLabel, size = 6, label.colour = NA) +
    annotate(geom = "richtext", x = limY.bee[2]+(limY.bee[2]/15), y = bottomX.bee,
             label = bottomLabel, size = 6, label.colour = NA)

## Saves

### DA Results

In [ ]:
write_csv(da_results.tnkilc.ctrlibd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ctrlibd.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.tnkilc.ccduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_ccduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.tnkilc.fgiduc, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgiduc.csv", 
                 thiscelltype, thiscelltype)
         )

In [ ]:
write_csv(da_results.tnkilc.fgidccd, 
         sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_fgidccd.csv", 
                 thiscelltype, thiscelltype)
         )

### Neighbourhoods

In [ ]:
Matrix::writeMM(nhoods(milo.tnkilc), 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_sparse.txt", 
                thiscelltype, thiscelltype)
       )

In [ ]:
write_tsv(as.data.frame(as.matrix(nhoodCounts(milo.tnkilc))) %>% rownames_to_column("nhood"),
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x_nhoods_dense.txt", 
                thiscelltype, thiscelltype)
       )

### Milo Object

In [ ]:
milo.tnkilc

In [ ]:
saveRDS(milo.tnkilc, 
        sprintf("../../predict/data/milo/%s/231226__predictCOL__CB_%s_T0_FGIDCCDUC_ds_milo_k200_10x.rds", thiscelltype, thiscelltype)
       )